# Treinamento com interface de alto nível

## Importação das bibliotecas

In [1]:
# http://pytorch.org/
from os.path import exists

import torch

In [2]:
import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.optim.lr_scheduler import StepLR

## Criação da rede

In [3]:
net_input = 32*32
net_output = 10 # 10 dígitos

In [4]:
class Net(nn.Module):
    def __init__(self, net_input, net_output):
        super(Net, self).__init__()
        self.layer1 = nn.Linear(net_input, 1024)
        self.layer2 = nn.Linear(1024, 2048)
        self.layer3 = nn.Linear(2048, 4096)
        self.layer4 = nn.Linear(4096, 8192)
        self.layer5 = nn.Linear(8192, 4096)
        self.layer6 = nn.Linear(4096, 2048)
        self.layer7 = nn.Linear(2048, 1024)
        self.layer8 = nn.Linear(1024, net_output)

    def forward(self, x):
        x = x.view(-1, 32*32)
        x = self.layer1(x)
        x = F.relu(x)
        x = self.layer2(x)
        x = F.relu(x)
        x = self.layer3(x)
        x = F.relu(x)
        x = self.layer4(x)
        x = F.relu(x)
        x = self.layer5(x)
        x = F.relu(x)
        x = self.layer6(x)
        x = F.relu(x)
        x = self.layer7(x)
        x = F.relu(x)
        x = self.layer8(x)
        output = F.log_softmax(x, dim=1)
        return output

model = Net(net_input, net_output)

In [5]:
print(model)

Net(
  (layer1): Linear(in_features=1024, out_features=1024, bias=True)
  (layer2): Linear(in_features=1024, out_features=2048, bias=True)
  (layer3): Linear(in_features=2048, out_features=4096, bias=True)
  (layer4): Linear(in_features=4096, out_features=8192, bias=True)
  (layer5): Linear(in_features=8192, out_features=4096, bias=True)
  (layer6): Linear(in_features=4096, out_features=2048, bias=True)
  (layer7): Linear(in_features=2048, out_features=1024, bias=True)
  (layer8): Linear(in_features=1024, out_features=10, bias=True)
)


## Treinamento

### Criando o objeto de treinamento

In [6]:
def train(log_interval, dry_run, model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % log_interval == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))
            if dry_run:
                break

In [7]:
def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item()  # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    acc = 100. * correct / len(test_loader.dataset)
    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        acc))
    return acc

## Avaliação

In [8]:
use_cuda = torch.cuda.is_available()

torch.manual_seed(1111)

device = torch.device("cuda" if use_cuda else "cpu")

train_kwargs = {'batch_size': 512}
test_kwargs = {'batch_size': 1024}
if use_cuda:
    cuda_kwargs = {'num_workers': 1,
                    'pin_memory': True,
                    'shuffle': True}
    train_kwargs.update(cuda_kwargs)
    test_kwargs.update(cuda_kwargs)

transform=transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.5], std = [0.5])
    ])
dataset1 = datasets.CIFAR10('../data', train=True, download=True,
                    transform=transform)
dataset2 = datasets.CIFAR10('../data', train=False,
                    transform=transform)
train_loader = torch.utils.data.DataLoader(dataset1,**train_kwargs)
test_loader = torch.utils.data.DataLoader(dataset2, **test_kwargs)

model = Net(net_input, net_output).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.999))

epochs = 5
scheduler = StepLR(optimizer, step_size=1, gamma=0.7)
best_acc = test(model, device, test_loader)

for epoch in range(1, epochs + 1):
    train(10, False, model, device, train_loader, optimizer, epoch)
    acc = test(model, device, test_loader)
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), "cifar10_cnn.pt")
    scheduler.step()

100%|██████████| 170M/170M [00:04<00:00, 34.1MB/s]



Test set: Average loss: 2.3027, Accuracy: 1000/10000 (10%)

Train Epoch: 1 [0/50000 (0%)]	Loss: 2.302193
Train Epoch: 1 [5120/50000 (10%)]	Loss: 2.213462
Train Epoch: 1 [10240/50000 (20%)]	Loss: 2.138023
Train Epoch: 1 [15360/50000 (31%)]	Loss: 2.130963
Train Epoch: 1 [20480/50000 (41%)]	Loss: 2.087342
Train Epoch: 1 [25600/50000 (51%)]	Loss: 2.045329
Train Epoch: 1 [30720/50000 (61%)]	Loss: 2.083711
Train Epoch: 1 [35840/50000 (71%)]	Loss: 2.040613
Train Epoch: 1 [40960/50000 (82%)]	Loss: 2.011012
Train Epoch: 1 [46080/50000 (92%)]	Loss: 2.002758

Test set: Average loss: 1.9680, Accuracy: 2554/10000 (26%)

Train Epoch: 2 [0/50000 (0%)]	Loss: 1.981909
Train Epoch: 2 [5120/50000 (10%)]	Loss: 1.895670
Train Epoch: 2 [10240/50000 (20%)]	Loss: 1.835878
Train Epoch: 2 [15360/50000 (31%)]	Loss: 1.913432
Train Epoch: 2 [20480/50000 (41%)]	Loss: 1.885082
Train Epoch: 2 [25600/50000 (51%)]	Loss: 1.914357
Train Epoch: 2 [30720/50000 (61%)]	Loss: 1.911359
Train Epoch: 2 [35840/50000 (71%)]	Loss: